# Muni New-Issue Pricer — all-in-one
This single notebook contains the complete system. **Run the cells in order once**: they install dependencies and write the code modules to disk next to this notebook. After that, only the *Price a wire* section is needed day to day.

You also need `ICE_Evals.csv` (the training data) in the same folder — upload it alongside this notebook, or produce it with the ICE pull described inside `run_pipeline.py`'s docs.

## 1. Dependencies (once per environment)

In [ ]:
%pip install -q pandas==3.0.5 numpy scikit-learn==1.9.0 lightgbm==4.7.0 joblib pyarrow

## 2. Write the code modules (once, or after edits here)

In [ ]:
%%writefile muni_model.py
"""
muni_model -- reconstructed muni new-issue pricing model.

Standalone reimplementation of the Curve_Analysis pipeline (which lives on the
b.fund machine), upgraded per the accuracy plan:

  * trains on the WHOLE filtered universe from ICE_Evals.csv, not one issuer
    (issuer becomes a feature via categoricals, not a filter)
  * gradient boosting (LightGBM) instead of a linear model
  * holdout evaluation with error reported in bps, bucketed by tenor /
    callability / coupon so you can see WHERE the model misses
  * a new-issue concession parameter to net out the secondary-eval vs
    primary-wire gap

Intended flow (see run_pipeline.py for the one-command version):

    df     = load_evals('ICE_Evals.csv')
    df     = clean_universe(df, state='California')
    bundle = train_yield_model(df)
    error_report(bundle, df)

    deal     = parse_wire(open('BAML Write-Up.txt').read())   # Wire_Parser
    template = template_from(df, issuer_contains='LOS ANG')
    results  = price_wire(deal, bundle, template)
"""

from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import joblib
except ImportError:
    joblib = None

from lightgbm import LGBMRegressor


# ---------------------------------------------------------------- day count

def days_30_360(start, end):
    """30/360 US day count between two dates (str 'm/d/Y' or datetime)."""
    if isinstance(start, str):
        start = datetime.strptime(start, '%m/%d/%Y')
    if isinstance(end, str):
        end = datetime.strptime(end, '%m/%d/%Y')
    d1, d2 = min(start.day, 30), end.day
    if d2 == 31 and d1 == 30:
        d2 = 30
    return (360 * (end.year - start.year)
            + 30 * (end.month - start.month)
            + (d2 - d1))


# ---------------------------------------------------------------- features

# numeric features, matching Run Code Part 1 where the eval file provides them
NUMERIC_FEATURES = [
    'current_coupon_rate', 'days_to_first_call_30_360', 'days_to_maturity_30_360',
    'days_to_next_call_30_360', 'days_to_refund_30_360',
    'days_from_call_to_maturity_30_360', 'issue_amount', 'next_call_price',
    'days_to_next_sink_date_30_360', 'next_sink_price', 'outstanding_amount',
    'principal_factor', 'log_issue_amount', 'log_outstanding_amount',
    'outstanding_pct_of_issue', 'call_moneyness', 'call_moneyness_pct',
    'above_call_price', 'par_dv01',
]
# par_dv01: analytic duration of a par bond with this coupon/maturity --
# computed from STRUCTURE only (identical for a wire tranche and a seasoned
# bond, no quotes involved), unlike ice_dv01 which derives from the bond's
# own bid/ask and is used only as a training FILTER (min_dv01).
# Tested and REJECTED as features (exp_config.py, 2026-08-19): issue_price,
# min_denom_amount, denom_increment_amount, bond_age_days, days_to_next_coupon.
# They left the universe holdout unchanged but degraded new-issue wire pricing
# ~3x (NYC TFA 3.3 -> 9.1 bps): their values for a brand-new bond (age 0,
# missing denominations, issue_price == today's price) sit outside the
# training distribution, so the model extrapolates badly exactly where we
# need it. Do not re-add without re-running exp_config.py on real wires.

CATEGORICAL_FEATURES = [
    'ad_valorem_tax_status_desc', 'bond_insurance', 'composite_rating',
    'coupon_type_desc', 'distinct_call_timing_desc', 'enhanced_composite_rating',
    'federal_tax_status_desc', 'incorporated_state_code_desc',
    'muni_security_type_desc', 'normalized_moody_enhanced_long_rating',
    'normalized_moody_long_rating', 'normalized_sandp_long_rating',
    'purpose_class_desc', 'purpose_sub_class_desc', 'security_class',
    'state_tax_status_desc', 'use_of_proceeds_desc', 'organization_master_id',
    'conduit_obligor_name_id', 'bank_qualified', 'call_notice', 'call_indicator',
    'called_redemption_type_desc',   # pre-refunded / escrowed / called status:
                                     # these trade off the escrow, not the credit
]


def load_evals(path):
    """Read an ICE evals CSV (as written by Pull_ICE_Evals.ps1 or the
    notebooks), indexed by CUSIP, with a 'target_yield' column added."""
    df = pd.read_csv(path, index_col=0, low_memory=False)

    # target: the feed's own mid yield when present, else bid/offer average
    yb = pd.to_numeric(df.get('yield_bid'), errors='coerce')
    yo = pd.to_numeric(df.get('yield_offer'), errors='coerce')
    ym = pd.to_numeric(df.get('yield_mid'), errors='coerce')
    df['target_yield'] = ym
    df.loc[df['target_yield'].isna(), 'target_yield'] = (yb + yo) / 2
    df.loc[df['target_yield'].isna(), 'target_yield'] = yb

    # eval-side DV01 proxy, same construction as the notebooks
    bid = pd.to_numeric(df.get('bid'), errors='coerce')
    offer = pd.to_numeric(df.get('offer'), errors='coerce')
    with np.errstate(divide='ignore', invalid='ignore'):
        df['ice_dv01'] = (offer - bid) / (yb - yo)

    return df


def clean_universe(df, federal_tax='Tax-exempt', state=None,
                   fixed_rate_only=True, drop_defaulted=True, min_dv01=None):
    """Filter to the modelable universe. Prints what each filter removes so
    nothing disappears silently."""
    n0 = len(df)
    mask = df['target_yield'].notna() & (df['target_yield'] > 0)
    print(f"no/zero yield:      -{(~mask).sum():,}")

    if federal_tax and 'federal_tax_status_desc' in df:
        m = df['federal_tax_status_desc'].eq(federal_tax)
        print(f"not {federal_tax}:   -{(mask & ~m).sum():,}")
        mask &= m
    if fixed_rate_only and 'coupon_type_desc' in df:
        m = df['coupon_type_desc'].eq('Fixed rate')
        print(f"not fixed rate:     -{(mask & ~m).sum():,}")
        mask &= m
    if drop_defaulted and 'default_indicator' in df:
        m = ~df['default_indicator'].astype(str).str.lower().isin(['true', '1', '1.0'])
        print(f"defaulted:          -{(mask & ~m).sum():,}")
        mask &= m
    if 'instrument_status_desc' in df:
        # Dead = matured / fully called / defunct -- their marks are stale and
        # poison both training and screening
        m = ~df['instrument_status_desc'].eq('Dead')
        print(f"dead instruments:   -{(mask & ~m).sum():,}")
        mask &= m
    if min_dv01 is not None and 'ice_dv01' in df:
        # quote-derived DV01 as a data-quality FILTER (the original Part 1
        # used min_dv01=1): drops ultra-short / oddly-quoted bonds whose
        # marks are noisiest. NaN dv01 (no two-sided quote) also drops.
        d = pd.to_numeric(df['ice_dv01'], errors='coerce')
        m = d >= min_dv01
        print(f"dv01 < {min_dv01} or unquoted: -{(mask & ~m.fillna(False)).sum():,}")
        mask &= m.fillna(False)
    if state and 'incorporated_state_code_desc' in df:
        m = df['incorporated_state_code_desc'].eq(state)
        print(f"not {state}:  -{(mask & ~m).sum():,}")
        mask &= m

    out = df[mask].copy()
    print(f"universe: {n0:,} -> {len(out):,}")
    return out


def stacked_frame(here=None):
    """THE canonical training frame: every dated snapshot in evals_archive
    stacked (fallback: the single current evals file). All trainers must use
    this -- three separate code paths once trained different recipes and two
    of them silently overwrote the production model (caught 8/27)."""
    here = Path(here) if here else Path(__file__).resolve().parent
    snaps = sorted((here / 'evals_archive').glob('ICE_Evals_*.csv'))
    if snaps:
        df = pd.concat([clean_universe(load_evals(p)) for p in snaps])
        print(f"stacked {len(snaps)} snapshot(s): {len(df):,} rows")
        return df
    return clean_universe(load_evals(here / 'ICE_Evals.csv'))


def build_features(df):
    """Derived columns (mirrors Run Code Part 1's derived features)."""
    df = df.copy()
    issue = pd.to_numeric(df.get('issue_amount'), errors='coerce')
    outst = pd.to_numeric(df.get('outstanding_amount'), errors='coerce')
    df['log_issue_amount'] = np.log1p(issue)
    df['log_outstanding_amount'] = np.log1p(outst)
    df['outstanding_pct_of_issue'] = outst / issue

    # analytic par-bond duration (points per 1% yield), pure structure:
    # ModDur_par = [1 - (1+c/2)^(-2T)] / c   with c = coupon, T = years
    cpn = pd.to_numeric(df.get('current_coupon_rate'), errors='coerce') / 100
    T = pd.to_numeric(df.get('days_to_maturity_30_360'), errors='coerce') / 360
    with np.errstate(all='ignore'):
        df['par_dv01'] = np.where(cpn > 0, (1 - (1 + cpn / 2) ** (-2 * T)) / cpn, T)

    mid = pd.to_numeric(df.get('mid'), errors='coerce')
    ncp = pd.to_numeric(df.get('next_call_price'), errors='coerce')
    df['call_moneyness'] = (mid - ncp).fillna(0.0)
    df['call_moneyness_pct'] = (df['call_moneyness'] / ncp).fillna(0.0)
    df['above_call_price'] = (mid > ncp).astype(float)

    # age and coupon-cycle position (skipped gracefully when cols are absent,
    # e.g. for wire tranches, where price_wire sets them directly)
    def _dt_col(name):
        return pd.to_datetime(df[name], errors='coerce') if name in df else None
    snap, issue_dt, nxt_cpn = _dt_col('model_date'), _dt_col('issue_date'), _dt_col('next_coupon_payment_date')

    # the feed's days_to_refund_30_360 arrives empty, but refund_date is
    # populated for ~8k pre-refunded/escrowed/called bonds -- rebuild the
    # feature so the model can price them off the escrow date
    refund_dt = _dt_col('refund_date')
    if refund_dt is not None and snap is not None:
        d2r = (pd.to_numeric(df['days_to_refund_30_360'], errors='coerce')
               if 'days_to_refund_30_360' in df else pd.Series(np.nan, index=df.index))
        if d2r.isna().all():
            df['days_to_refund_30_360'] = (refund_dt - snap).dt.days
    if 'bond_age_days' not in df:
        df['bond_age_days'] = ((snap - issue_dt).dt.days
                               if snap is not None and issue_dt is not None else np.nan)
    if 'days_to_next_coupon' not in df:
        df['days_to_next_coupon'] = ((nxt_cpn - snap).dt.days
                                     if snap is not None and nxt_cpn is not None else np.nan)
    return df


def _canon_cat(s):
    """Canonical string form for a categorical column, so training and
    prediction always agree: any numeric-looking value is rendered as a float
    string ('9' and 9 and 9.0 all become '9.0'); everything else is str().
    Without this, a wire-supplied rating of 9 (str '9') would not match the
    training category '9.0' and would silently turn into a missing value."""
    out = pd.Series(s).astype(str)
    as_num = pd.to_numeric(out, errors='coerce')
    mask = as_num.notna()
    return out.where(~mask, as_num.astype(float).astype(str))


def _prep_matrix(df, numeric, categorical, categories=None):
    """Assemble the model matrix; categoricals as pandas category dtype so
    LightGBM handles them natively. `categories` (from training) pins the
    category sets at prediction time."""
    X = pd.DataFrame(index=df.index)
    for c in numeric:
        X[c] = pd.to_numeric(df[c], errors='coerce') if c in df else np.nan
    for c in categorical:
        if c in df:
            s = _canon_cat(df[c])
            s.index = df.index
        else:
            s = pd.Series('missing', index=df.index)
        if categories is not None:
            X[c] = pd.Categorical(s, categories=categories[c])
        else:
            X[c] = s.astype('category')
    return X


# ---------------------------------------------------------------- training

def predict_bundle(bundle, X):
    """Average prediction across the bundle's ensemble members."""
    models = bundle.get('models') or [bundle['model']]
    return np.mean([m.predict(X) for m in models], axis=0)


def train_yield_model(df, numeric=None, categorical=None,
                      test_size=0.2, seed=42, n_ensemble=3, **lgbm_kwargs):
    """Train the yield model on cleaned+featured evals. Returns a bundle dict
    with the ensemble, feature lists, pinned categories, holdout frame, and
    holdout MAE in bps.

    Uses the L1 (absolute error) objective -- the metric we actually care
    about -- with early stopping on a validation slice carved out of the
    training portion (the test holdout stays untouched), and averages
    n_ensemble models trained with different seeds.

    NOTE: the split is random because one eval pull is a single cross-section.
    Once daily pulls accumulate, switch to time-based splits (train on past
    days, test on the latest) -- random splits on panel data flatter the model.
    """
    import lightgbm as lgb

    # sub-1-year bonds are excluded from TRAINING (adopted 8/26 after
    # exp_shortdrop/exp_dv01/exp_combo): their erratic marks polluted split
    # decisions for the whole curve -- dropping them cut the >=1yr holdout
    # median from 2.24 to 0.81 bps with no net wire degradation. Consequence:
    # the model has no support under 1yr, so price_wire marks sub-1yr
    # tranches as reduced confidence.
    if 'days_to_maturity_30_360' in df:
        tenor = pd.to_numeric(df['days_to_maturity_30_360'], errors='coerce')
        n0 = len(df)
        df = df[tenor >= 360]
        print(f"training filter: dropped {n0 - len(df):,} sub-1yr bonds -> {len(df):,}")

    df = build_features(df)

    numeric = [c for c in (numeric or NUMERIC_FEATURES) if c in df.columns]
    categorical = [c for c in (categorical or CATEGORICAL_FEATURES) if c in df.columns]
    missing = sorted(set(NUMERIC_FEATURES + CATEGORICAL_FEATURES)
                     - set(numeric) - set(categorical))
    if missing:
        print(f"features not in eval file (skipped): {missing}")

    X = _prep_matrix(df, numeric, categorical)
    y = df['target_yield'].astype(float)

    # CUSIP-level split (8/27 fix): stacked snapshots repeat CUSIPs, so the
    # old row-level split let ~80% of "held-out" bonds also reach training
    # via their other snapshot's row. Split by CUSIP, and persist the set in
    # the bundle so downstream evaluations grade on genuinely unseen bonds.
    cusips = pd.Index(df.index.unique())
    rc = np.random.RandomState(seed).rand(len(cusips))
    test_cusips = set(cusips[rc < test_size])
    val_cusips = set(cusips[(rc >= test_size) & (rc < test_size + 0.08)])
    test_mask = df.index.isin(test_cusips)
    val_mask = df.index.isin(val_cusips)
    fit_mask = ~test_mask & ~val_mask

    params = dict(objective='l1', n_estimators=8000, learning_rate=0.03,
                  num_leaves=127, min_child_samples=30, subsample=0.9,
                  subsample_freq=1, colsample_bytree=0.9, verbose=-1)
    params.update(lgbm_kwargs)

    models = []
    for k in range(n_ensemble):
        m = LGBMRegressor(**{**params, 'random_state': seed + k})
        m.fit(X[fit_mask], y[fit_mask],
              eval_set=[(X[val_mask], y[val_mask])], eval_metric='l1',
              callbacks=[lgb.early_stopping(150, verbose=False)])
        models.append(m)

    bundle = {'models': models, 'model': models[0],
              'numeric': numeric, 'categorical': categorical,
              'holdout_cusips': sorted(test_cusips),
              'categories': {c: list(X[c].cat.categories) for c in categorical}}

    pred = predict_bundle(bundle, X[test_mask])
    err_bps = (pred - y[test_mask].to_numpy()) * 100
    mae = float(np.mean(np.abs(err_bps)))
    med = float(np.median(np.abs(err_bps)))
    print(f"holdout ({test_mask.sum():,} bonds): MAE {mae:.1f} bps | median {med:.1f} bps "
          f"| ensemble of {n_ensemble}, best_iter {[m.best_iteration_ for m in models]}")

    holdout = df[test_mask].copy()
    holdout['pred_yield'] = pred
    holdout['err_bps'] = err_bps
    bundle.update({'holdout': holdout, 'mae_bps': mae, 'median_bps': med})
    return bundle


def error_report(bundle, top_n=12):
    """Where does the model miss? Buckets holdout error by tenor, callability,
    and coupon. This is what tells you what to fix next."""
    h = bundle['holdout']
    print("\n--- holdout error by bucket (mean abs bps / count) ---")

    tenor_yrs = pd.to_numeric(h.get('days_to_maturity_30_360'), errors='coerce') / 360
    buckets = pd.cut(tenor_yrs, [0, 2, 5, 10, 15, 20, 30, 100])
    print("\nby tenor (yrs):")
    print(h.groupby(buckets, observed=True)['err_bps']
          .agg(mae=lambda s: s.abs().mean(), n='count').round(1))

    if 'call_indicator' in h:
        print("\nby callability:")
        print(h.groupby(h['call_indicator'].astype(str))['err_bps']
              .agg(mae=lambda s: s.abs().mean(), n='count').round(1))

    cpn = pd.to_numeric(h.get('current_coupon_rate'), errors='coerce')
    print("\nby coupon:")
    print(h.groupby(pd.cut(cpn, [0, 3, 4, 5, 6, 20]), observed=True)['err_bps']
          .agg(mae=lambda s: s.abs().mean(), n='count').round(1))

    print(f"\nworst {top_n} holdout misses:")
    cols = [c for c in ['target_yield', 'pred_yield', 'err_bps',
                        'current_coupon_rate', 'days_to_maturity_30_360'] if c in h]
    print(h.reindex(h['err_bps'].abs().sort_values(ascending=False).index)[cols].head(top_n).round(2))


def confidence_flags(df, pred_yield, ens_std_bps):
    """Trust filter for using model-vs-market gaps as SIGNALS (secondary
    screening). Every flag is computable before knowing the true yield.

    Validated on the seed-42 holdout (diag_confidence.py, 2026-08-19):
    trusted 65% of bonds -> mean err 3.5 / median 1.7 bps; the filter caught
    89% of all >50bp errors and 95% of >100bp errors in advance. A large
    model-vs-market gap on a FLAGGED bond usually means the market knows
    something the model can't see (distress, escrow, illiquidity) -- treat it
    as a research lead, never a trade signal.

    Returns (flags DataFrame, trusted boolean Series).
    """
    idx = df.index
    flags = pd.DataFrame(index=idx)
    rating = (pd.to_numeric(df['composite_rating'], errors='coerce')
              if 'composite_rating' in df else pd.Series(np.nan, index=idx))
    flags['unrated'] = rating.isna()
    flags['junk_rated'] = rating > 10
    flags['pred_yield_high'] = pd.Series(np.asarray(pred_yield), index=idx) > 5.5
    flags['conduit'] = (df['conduit_obligor_name_id'].notna()
                        if 'conduit_obligor_name_id' in df else False)
    flags['micro'] = (pd.to_numeric(df['outstanding_amount'], errors='coerce') < 2_000_000
                      if 'outstanding_amount' in df else False)
    flags['ens_disagree'] = pd.Series(np.asarray(ens_std_bps), index=idx) > 5
    flags['ultra_short'] = (pd.to_numeric(df['days_to_maturity_30_360'], errors='coerce') < 540
                            if 'days_to_maturity_30_360' in df else False)
    return flags, ~flags.any(axis=1)


def save_bundle(bundle, path):
    slim = {k: v for k, v in bundle.items() if k != 'holdout'}
    joblib.dump(slim, path)
    print(f"model saved -> {path}")


def load_bundle(path):
    return joblib.load(path)


# ---------------------------------------------------------------- wire pricing

def template_from(df, issuer_contains=None, cusip=None):
    """Categorical defaults for a new issue, taken from comparable existing
    bonds: a specific CUSIP, or the modal values across all bonds whose
    issuer/obligor text matches `issuer_contains`."""
    if cusip is not None:
        row = df.loc[cusip]
        if isinstance(row, pd.DataFrame):   # duplicated index entry
            row = row.iloc[0]
        return {c: str(row[c]) for c in CATEGORICAL_FEATURES if c in df.columns}

    pool = df
    if issuer_contains:
        text_cols = [c for c in ['primary_name_abbreviated', 'organization_master_id',
                                 'conduit_obligor_name_id'] if c in df.columns]
        m = pd.Series(False, index=df.index)
        for c in text_cols:
            m |= df[c].astype(str).str.contains(issuer_contains, case=False, na=False)
        if m.any():
            pool = df[m]
            print(f"template: {m.sum():,} bonds match issuer '{issuer_contains}'")
        else:
            print(f"template: no issuer match for '{issuer_contains}', using full-universe modes")
    return {c: str(pool[c].mode(dropna=False).iloc[0])
            for c in CATEGORICAL_FEATURES if c in pool.columns}


def price_wire(deal, bundle, template, settlement_date=None,
               concession_bps=0.0, rating_overrides=True):  # 0.0 = API default: caller supplies calibration
    """Price every tranche of a parsed wire (Wire_Parser.parse_wire output).
    Returns a DataFrame of wire vs model yields with error in bps.

    concession_bps: expected new-issue concession -- added to the model's
    secondary-market yield before comparing to the wire. Estimate it from the
    archive once a few deals accumulate; 0 until then.
    """
    settlement = settlement_date or deal['dated_date']
    if not settlement:
        settlement = datetime.now().strftime('%m/%d/%Y')
        print(f"WARNING: wire has no DATED line and no settlement_date passed -- "
              f"assuming today ({settlement}). Pass --settlement for precision.")
    call_date, call_price = deal['call_date'], deal['call_price']

    # guard: if the wire's rating is far from the template pool's rating, the
    # template is probably anchored to a DIFFERENT credit of the same issuer
    # (e.g. an issuer's AA sales-tax bonds as template for its BBB toll deal).
    # The model leans on issuer identity, so a credit-mismatched template
    # skews every tranche the same direction.
    tmpl_moody = pd.to_numeric(pd.Series([template.get('normalized_moody_long_rating')]),
                               errors='coerce').iloc[0]
    if deal.get('moody_rating') is not None and pd.notna(tmpl_moody):
        gap = abs(deal['moody_rating'] - tmpl_moody)
        if gap >= 3:
            print(f"WARNING: wire rating ({deal['moody_rating']}) is {gap:.0f} notches from "
                  f"the template pool's typical rating ({tmpl_moody:.0f}). The template may be "
                  f"a different credit of this issuer -- consider a broader --issuer match "
                  f"or a --template-cusip from a comparable credit.")

    rows = []
    for t in deal['tranches']:
        if t['price'] is None:
            continue
        dtm = days_30_360(settlement, t['maturity'])
        callable_ = (call_date is not None and
                     datetime.strptime(t['maturity'], '%m/%d/%Y')
                     > datetime.strptime(call_date, '%m/%d/%Y'))
        dtc = days_30_360(settlement, call_date) if callable_ else np.nan

        feat = dict(template)
        feat.update({
            'current_coupon_rate': t['coupon'],
            'days_to_maturity_30_360': dtm,
            'days_to_first_call_30_360': dtc,
            'days_to_next_call_30_360': dtc,
            'days_from_call_to_maturity_30_360': dtm - dtc if callable_ else np.nan,
            'issue_amount': deal['issue_amount'],
            'outstanding_amount': t['amount'],
            'next_call_price': call_price if callable_ else np.nan,
            'principal_factor': 1.0,
            'mid': t['price'],
            'call_indicator': str(callable_),
            'issue_price': t['price'],
            'bond_age_days': 0.0,
            'days_to_next_coupon': (days_30_360(settlement, deal['first_coupon'])
                                    if deal.get('first_coupon') else np.nan),
        })
        # only override template ratings if the wire actually carried ratings
        if rating_overrides and deal.get('moody_rating') is not None:
            feat['normalized_moody_long_rating'] = str(deal['moody_rating'])
        if rating_overrides and deal.get('sandp_rating') is not None:
            feat['normalized_sandp_long_rating'] = str(deal['sandp_rating'])
        rows.append(feat)

    if not rows:
        raise ValueError(
            "No priced tranches parsed from this wire. Usual causes: the "
            "WIRE_TEXT cell still contains the placeholder text, the paste "
            "was incomplete, or this dealer's format needs a parser "
            "extension. Check the wire text and re-run.")
    fdf = build_features(pd.DataFrame(rows))
    X = _prep_matrix(fdf, bundle['numeric'], bundle['categorical'], bundle['categories'])
    # version guard: if the loaded model expects a derived feature that this
    # code version didn't compute (all-NaN), predictions would silently
    # collapse (seen 8/26: stale kernel + new model -> flat curve, -180bp
    # errors). Refuse loudly instead.
    for _c in ('par_dv01', 'call_moneyness', 'log_issue_amount'):
        if _c in bundle['numeric'] and _c in X and X[_c].isna().all():
            raise RuntimeError(
                f"MODEL/CODE VERSION MISMATCH: the model expects feature '{_c}' "
                f"but this code produced none. You are running STALE code "
                f"against a newer model.joblib -- restart the Jupyter kernel "
                f"(or importlib.reload(muni_model)), or use Price Wire.bat.")
    members = np.array([m.predict(X) for m in (bundle.get('models') or [bundle['model']])])
    # the new-issue concession lives in the LONG bonds -- short tranches get
    # almost none (observed on PSU and Portland: +18-22bp false "rich" reads
    # at 1yr under a flat concession). Ramp: zero at 0yr, full from 8yrs.
    # Keyed on years to WORKOUT (call date if priced-to-call, else maturity)
    # -- the same axis the calibration measures on (unified 8/27).
    workout_days = np.array([
        days_30_360(settlement, t['ptc_date'] or t['maturity'])
        for t in deal['tranches'] if t['price'] is not None], dtype=float)
    ramp = np.clip(workout_days / 360.0 / 8.0, 0, 1)
    pred = members.mean(axis=0) + concession_bps * ramp / 100
    ens_std_bps = members.std(axis=0) * 100

    priced = [t for t in deal['tranches'] if t['price'] is not None]
    out = pd.DataFrame({
        'Maturity': [t['maturity'] for t in priced],
        'Coupon': [t['coupon'] for t in priced],
        'Amount ($)': [t['amount'] for t in priced],
        'Priced To': [t['ptc_date'] or 'Maturity' for t in priced],
        'Wire Yield': [t['yield'] for t in priced],
        'Wire Price': [t['price'] for t in priced],
        'Model Yield': np.round(pred, 4),
    }).set_index('Maturity')
    out['Error (bps)'] = ((out['Model Yield'] - out['Wire Yield']) * 100).round(1)
    # decomposition: total error = deal LEVEL (this deal's concession vs the
    # calibrated estimate -- irreducible ~+/-2bp until the deal count grows)
    # + tranche SHAPE (relative value within the deal, the model's strength).
    # Deal-Rel = error minus the deal's long-end mean level: which tranches
    # are rich/cheap RELATIVE TO THIS DEAL'S OWN PRICING LEVEL.
    long_end = ramp >= 0.999
    if long_end.sum() >= 3:
        level = float(out['Error (bps)'].to_numpy()[long_end].mean())
        out['Deal-Rel (bps)'] = (out['Error (bps)'] - level).round(1)
    # confidence: disagreement between ensemble members (high std = off the
    # training map), AND tranches under ~1.25yr -- the model trains on no
    # sub-1yr bonds (see train_yield_model), so its front-end reads are
    # extrapolation and get flagged rather than oversold.
    out['Model Std (bps)'] = np.round(ens_std_bps, 1)
    tenor_days = pd.to_numeric(fdf['days_to_maturity_30_360'], errors='coerce').to_numpy()
    out['Confidence'] = np.where((ens_std_bps <= 5) & (tenor_days >= 450), 'high', 'CHECK')

    err = out['Error (bps)'].abs()
    print(f"wire vs model: mean abs error {err.mean():.1f} bps | "
          f"median {err.median():.1f} | max {err.max():.1f}  "
          f"(concession {concession_bps:+.1f} bps)")
    return out


In [ ]:
%%writefile Wire_Parser.py
"""
Wire_Parser -- parse a dealer new-issue pricing wire (the standard format in
'BAML Write-Up.txt') and run EVERY tranche through the ICE / Spline yield and
DV01 models, comparing model yield to the dealer's yield in bps.

This replaces the hand-typed inputs of 'Run Code Part 2.txt'. The wire format
this expects:

    RE: $ 509,145,000*
    <issuer description lines>
    MOODY'S: Baa2 (Stable)                  S&P:   NR
    FITCH:   BBB- (Stable)                  KROLL: NR
    DATED:09/03/2026   FIRST COUPON:12/01/2026
    06/01/2034      5,650M     5.00%     3.57      0.40
                          (Approx. $ Price 109.595)
    06/01/2037      8,600M     5.00%     3.90      0.40
                          (Approx. $ Price PTC 06/01/2036 108.840)
    ...
    CALL FEATURES:  Optional call in 06/01/2036 @ 100.00

Usage (see 'Run Code Part 3.txt'; requires the models from Run Code Part 1
plus predict_curve / days_30_360 already in the namespace):

    deal    = parse_wire(open(path).read())
    results = price_wire_tranches(deal, df,
                                  ice_yield_model, spline_yield_model,
                                  ice_dv01_model, spline_dv01_model)

parse_wire() is pure stdlib text -> dict, so it is trivially testable on any
new wire before the models ever get involved.
"""

import re
from datetime import datetime

import pandas as pd

# Numeric scales matching ratings_dict in ICE_Data_Pull (1 = AAA ... 22 = D).
MOODY_SCALE = {'Aaa': 1, 'Aa1': 2, 'Aa2': 3, 'Aa3': 4, 'A1': 5, 'A2': 6, 'A3': 7,
               'Baa1': 8, 'Baa2': 9, 'Baa3': 10, 'Ba1': 11, 'Ba2': 12, 'Ba3': 13,
               'B1': 14, 'B2': 15, 'B3': 16, 'Caa1': 17, 'Caa2': 18, 'Caa3': 19,
               'Ca': 20, 'C': 21}
SP_SCALE = {'AAA': 1, 'AA+': 2, 'AA': 3, 'AA-': 4, 'A+': 5, 'A': 6, 'A-': 7,
            'BBB+': 8, 'BBB': 9, 'BBB-': 10, 'BB+': 11, 'BB': 12, 'BB-': 13,
            'B+': 14, 'B': 15, 'B-': 16, 'CCC+': 17, 'CCC': 18, 'CCC-': 19,
            'CC': 20, 'C': 21, 'D': 22}

# maturity row, tolerant of common dialect variations:
#   - optional 1-2 letter flag after the date (sinking fund 'S', term 'T')
#   - optional "orders / +spread" token (Loop Capital: "78,475 / +40")
#   - optional '+' after the yield (priced-to-premium-call stub)
#   - trailing extras (same-line price parenthetical, ratings text)
# The specific anchors (date + amount'M' + coupon'%' + yield) keep narrative
# lines from matching; the tranche-sum check catches anything that slips.
MATURITY_ROW = re.compile(
    r'^\s*(\d{2}/\d{2}/\d{4})(?:\s+[A-Z]{1,2}(?=\s))?\s+(?:[\d,]+M?\s*/\s*\+?-?\d+\s+)?'
    r'([\d,]+)M\s+([\d.]+)%\s+([\d.]+)\+?(?:\s+([\d.]+))?(?:\s+\S.*)?$')
# note the M? in the orders token: Wells Fargo prints "2,500M / +7" (amount
# with M before the slash), Loop prints "78,475 / +40" (without)
PRICE_ROW = re.compile(
    r'\(Approx\.\s*\$\s*Price\s*(?:PTC\s*(\d{2}/\d{2}/\d{4})\s*)?([\d.]+)')
# no closing \) required: Wells Fargo wraps price lines mid-parenthesis
# ("...107.266 Approx.\nYTM 4.387)") -- the price is captured before the wrap


def _date(s):
    return datetime.strptime(s, '%m/%d/%Y')


def _rating(text, agency):
    m = re.search(re.escape(agency) + r"\s*:\s*(\S+)", text)
    if not m:
        return 'NR'
    raw = m.group(1)
    # a blank rating slot makes \S+ swallow the NEXT label (seen live:
    # "FITCH:            KROLL: NR" parsed Fitch as "KROLL:")
    if raw.endswith(':'):
        return 'NR'
    # real wires carry compound formats -- "Aa1/VMIG-1" (long/short dual
    # rating), "AA+*" (watch flag). Take the long-term component and strip
    # decorations before validating against the agency scales.
    val = raw.split('/')[0].rstrip('*').strip()
    if val in MOODY_SCALE or val in SP_SCALE or val.upper() == 'NR':
        return val
    print(f"NOTE: unrecognized {agency} rating '{raw}' on this wire -- "
          f"treating as NR (template rating will be used)")
    return 'NR'


def parse_wire(text):
    """Parse one pricing wire into deal-level fields + a list of tranches."""
    deal = {}
    lines = text.splitlines()

    m = re.search(r'RE:\s*\$\s*([\d,]+)', text)
    if not m:
        # no "RE: $" header (some dealers put the size in the Subject line) --
        # fall back to the first large dollar amount anywhere in the text
        m = re.search(r'\$\s*([\d,]{9,})', text)
    deal['issue_amount'] = int(m.group(1).replace(',', '')) if m else None

    # issuer description: the non-blank lines following the RE: line
    desc = []
    for j, line in enumerate(lines):
        if 'RE:' in line:
            for nxt in lines[j + 1:]:
                s = nxt.strip()
                if not s:
                    if desc:
                        break
                    continue
                desc.append(s)
            break
    deal['description'] = ' '.join(desc)
    if not deal['description']:
        m = re.search(r'Subject\s+(.+)', text)
        deal['description'] = m.group(1).strip() if m else 'UNKNOWN DEAL'

    deal['moody'] = _rating(text, "MOODY'S")
    deal['sandp'] = _rating(text, 'S&P')
    deal['fitch'] = _rating(text, 'FITCH')

    # Numeric ratings for the model; if an agency is NR fall back to the
    # others so the model always gets a number (S&P NR -> use Fitch, etc.)
    moody_num = MOODY_SCALE.get(deal['moody'])
    sandp_num = SP_SCALE.get(deal['sandp'])
    fitch_num = SP_SCALE.get(deal['fitch'])
    deal['moody_rating'] = moody_num if moody_num is not None else (sandp_num or fitch_num)
    deal['sandp_rating'] = sandp_num if sandp_num is not None else (fitch_num or moody_num)

    m = re.search(r'DATED\s*:\s*(\d{2}/\d{2}/\d{4})', text)
    deal['dated_date'] = m.group(1) if m else None
    m = re.search(r'FIRST\s+COUPON\s*:\s*(\d{2}/\d{2}/\d{4})', text)
    deal['first_coupon'] = m.group(1) if m else None

    m = re.search(r'call\s+in\s+(\d{2}/\d{2}/\d{4})\s*@\s*([\d.]+)', text, re.IGNORECASE)
    deal['call_date'] = m.group(1) if m else None
    deal['call_price'] = float(m.group(2)) if m else 100.0

    tranches = []
    for i, line in enumerate(lines):
        m = MATURITY_ROW.match(line)
        if not m:
            continue
        tranche = {
            'maturity': m.group(1),
            'amount': int(m.group(2).replace(',', '')) * 1000,  # wire prints $ thousands
            'coupon': float(m.group(3)),
            'yield': float(m.group(4)),
            'takedown': float(m.group(5)) if m.group(5) else None,
            'price': None,
            'ptc_date': None,
        }
        # the (Approx. $ Price ...) is usually on a following line, but some
        # dialects put it at the end of the maturity row itself
        for nxt in lines[i:i + 3]:
            pm = PRICE_ROW.search(nxt)
            if pm:
                tranche['ptc_date'] = pm.group(1)
                tranche['price'] = float(pm.group(2))
                break
        tranches.append(tranche)
    deal['tranches'] = tranches

    total = sum(t['amount'] for t in tranches)
    if deal['issue_amount'] and total != deal['issue_amount']:
        print(f"WARNING: tranche amounts sum to {total:,} "
              f"vs stated issue size {deal['issue_amount']:,} -- check the parse.")
    return deal


def price_wire_tranches(deal, df,
                        ice_yield_model, spline_yield_model,
                        ice_dv01_model, spline_dv01_model,
                        settlement_date=None, template_cusip=None,
                        output_path=None):
    """
    Run every tranche of a parsed wire through the four models. Returns a
    DataFrame with model yields vs the dealer's wire yields, error in bps,
    and predicted DV01s. Prints a mean/median/max abs-error summary -- the
    number to beat is ICE's ~5bp.

    df and the four models are the outputs of build_ice_spline_curves.
    Needs predict_curve and days_30_360 in the namespace (%run the
    Curve_Analysis / New_Holidays notebooks first).

    template_cusip: existing bond of this issuer whose categorical features
    (state, sector, security type...) stand in for the new issue. Defaults
    to df.index[0] -- fine while df is filtered to one issuer, but pass it
    explicitly if the filter ever widens.
    """
    settlement = settlement_date or deal['dated_date']
    call_date, call_price = deal['call_date'], deal['call_price']
    cusip = template_cusip if template_cusip is not None else df.index[0]

    rows = []
    for t in deal['tranches']:
        if t['price'] is None:
            print(f"Skipping {t['maturity']} {t['coupon']}% -- no dollar price parsed.")
            continue

        days_to_maturity = days_30_360(settlement, t['maturity'])

        # callable only if it matures AFTER the call date (the wire's PTC tag
        # marks priced-to-call, not callability -- a discount tranche past the
        # call date is still callable but priced to maturity)
        callable_ = call_date is not None and _date(t['maturity']) > _date(call_date)
        if callable_:
            days_to_call = days_30_360(settlement, call_date)
            days_call_to_mat = days_to_maturity - days_to_call
            call_moneyness = t['price'] - call_price
            call_moneyness_pct = call_moneyness / call_price
            above_call = float(t['price'] > call_price)
            next_call = call_price
        else:
            days_to_call = ''
            days_call_to_mat = ''
            call_moneyness, call_moneyness_pct, above_call = 0.0, 0.0, 0.0
            next_call = 100.0

        base = {
            'current_coupon_rate': t['coupon'],
            'call_indicator': str(callable_),
            'days_to_maturity_30_360': days_to_maturity,
            'days_to_next_call_30_360': days_to_call,
            'days_to_first_call_30_360': days_to_call,
            'days_from_call_to_maturity_30_360': days_call_to_mat,
            'issue_amount': deal['issue_amount'],
            'outstanding_amount': t['amount'],
            'next_call_price': next_call,
        }

        # NOTE: the DV01 feature lists use 'current_coupon' (not
        # 'current_coupon_rate'), and the Spline DV01 model uses 'Spline Mid'
        # (not 'mid') -- Run Code Part 2 set the wrong keys, so the coupon and
        # price never reached those models. Correct per-model keys here.
        ice_dv01_changes = {**base, 'current_coupon': t['coupon'],
                            'mid': t['price'],
                            'ICE call_moneyness': call_moneyness,
                            'ICE call_moneyness_pct': call_moneyness_pct,
                            'ICE above_call_price': above_call}
        spline_dv01_changes = {**base, 'current_coupon': t['coupon'],
                               'Spline Mid': t['price'],
                               'Spline call_moneyness': call_moneyness,
                               'Spline call_moneyness_pct': call_moneyness_pct,
                               'Spline above_call_price': above_call}

        ice_dv01 = predict_curve(model=ice_dv01_model[0], result=ice_dv01_model[1],
                                 cusip=cusip, **ice_dv01_changes)
        spline_dv01 = predict_curve(model=spline_dv01_model[0], result=spline_dv01_model[1],
                                    cusip=cusip, **spline_dv01_changes)

        yield_base = {**base,
                      'normalized_moody_long_rating': deal['moody_rating'],
                      'normalized_sandp_long_rating': deal['sandp_rating']}
        ice_yield_changes = {**yield_base,
                             'ICE call_moneyness': call_moneyness,
                             'ICE call_moneyness_pct': call_moneyness_pct,
                             'ICE above_call_price': above_call}
        spline_yield_changes = {**yield_base,
                                'Spline call_moneyness': call_moneyness,
                                'Spline call_moneyness_pct': call_moneyness_pct,
                                'Spline above_call_price': above_call}

        ice_yld = predict_curve(ice_yield_model[0], ice_yield_model[1],
                                cusip='', **ice_yield_changes)
        spline_yld = predict_curve(spline_yield_model[0], spline_yield_model[1],
                                   cusip='', **spline_yield_changes)

        rows.append({
            'Maturity': t['maturity'],
            'Coupon': t['coupon'],
            'Amount ($)': t['amount'],
            'Priced To': t['ptc_date'] or 'Maturity',
            'Wire Yield': t['yield'],
            'Wire Price': t['price'],
            'ICE Yield': round(ice_yld, 4),
            'Spline Yield': round(spline_yld, 4),
            'ICE Error (bps)': round((ice_yld - t['yield']) * 100, 1),
            'Spline Error (bps)': round((spline_yld - t['yield']) * 100, 1),
            'ICE DV01': round(ice_dv01, 4),
            'Spline DV01': round(spline_dv01, 4),
        })

    results = pd.DataFrame(rows).set_index('Maturity')

    for name in ('ICE', 'Spline'):
        err = results[f'{name} Error (bps)'].abs()
        print(f"{name:6s}: mean abs error {err.mean():.1f} bps | "
              f"median {err.median():.1f} | max {err.max():.1f}")

    if output_path:
        results.to_csv(output_path)
    return results


In [ ]:
%%writefile run_pipeline.py
"""
run_pipeline -- the automated path: Bloomberg wire text in, priced deal out.

    python run_pipeline.py --wire "BAML Write-Up.txt" --evals ICE_Evals.csv

What it does:
  1. trains the yield model from the evals CSV (or loads model.joblib if it
     exists -- pass --retrain to force a rebuild after a fresh eval pull)
  2. parses the wire (Wire_Parser)
  3. prices every tranche, prints wire-vs-model error in bps
  4. archives the result to wire_archive/<deal>_<timestamp>.csv -- the
     accumulating scoreboard for grinding error down below ICE's ~5bp

Options:
  --state "California"        train on one state's bonds (default: all)
  --issuer "LOS ANG"          issuer text for the categorical template
  --concession 3              new-issue concession in bps to add to model yield
  --retrain                   rebuild the model even if model.joblib exists
  --report                    print the bucketed holdout error report
"""

import argparse
import sys
from datetime import datetime
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))

from Wire_Parser import parse_wire  # noqa: E402
import muni_model as mm  # noqa: E402


def auto_issuer(description, df, min_frac=0.5, min_hits=3):
    """Match the wire's issuer description to an ICE issuer name without a
    hand-typed --issuer. ICE names are abbreviated ('LOS ANG CY CA MET TRA
    AUT'), so score each name by the fraction of its tokens that are prefixes
    of words in the description. Returns the best name or None."""
    import re
    if 'primary_name_abbreviated' not in df.columns:
        return None
    words = {w for w in re.findall(r'[A-Za-z]{2,}', description.upper())}
    best_name, best_key = None, (0, 0.0, 0)
    counts = df['primary_name_abbreviated'].astype(str).str.upper().value_counts()

    SHORT_SYNONYMS = {'ST': {'ST', 'STATE'}, 'CY': {'CY', 'CITY'},
                      'CO': {'CO', 'COUNTY'}}

    def _tok_match(t, w):
        # short tokens (ST, RE, NY...) prefix-match half the dictionary
        # ("RE" -> RELEASE), which once matched a 1-bond Maryland parking
        # issuer to Penn State. Short tokens must match exactly -- except a
        # few standard ICE abbreviations with one canonical expansion
        # ('CALIFORNIA ST' must match "STATE OF CALIFORNIA", caught on the
        # CA GO deal 8/27). 3+ letter tokens may match as prefixes.
        if len(t) >= 3:
            return w.startswith(t)
        return w in SHORT_SYNONYMS.get(t, {t})

    for name, n_bonds in counts.items():
        if n_bonds < 5:
            continue   # a handful of bonds is too thin to be a pricing context
        toks = [t for t in name.split() if len(t) >= 2]
        # 2-token names are legitimate (state GOs: 'WASHINGTON ST') and
        # all-short-token names are too ('LOS ANG CY CA MET TRA AUT') --
        # both were excluded by earlier gates (caught 8/27, unseen-input
        # test + regression). Guard against junk differently: short names
        # must match COMPLETELY (frac == 1.0), and hits required scales
        # with name length instead of a flat 3.
        if len(toks) < 2:
            continue
        # primary score: the CHARACTER MASS of distinct wire words this name
        # accounts for -- distinctive words (PENNSYLVANIA) outweigh filler
        # (THE, OF), which once let Rutgers outscore Penn State's own name
        # by covering four junk words. Stopwords count for nothing.
        STOP = {'THE', 'OF', 'AND', 'FOR', 'SERIES', 'BONDS', 'STATE', 'ST'}
        long_toks = [t for t in toks if len(t) >= 3]
        covered = {w for w in words if any(_tok_match(t, w) for t in long_toks)}
        cover_mass = sum(len(w) for w in covered if w not in STOP)
        hits = sum(any(_tok_match(t, w) for w in words) for t in toks)
        frac = hits / len(toks)
        key = (cover_mass, frac, n_bonds)
        need_hits = min(min_hits, len(toks))
        need_frac = 1.0 if len(toks) == 2 else min_frac
        if frac >= need_frac and hits >= need_hits and key > best_key:
            best_name, best_key = name, key
    if best_name:
        print(f"auto-matched issuer: '{best_name}' ({best_key[2]:,} bonds, "
              f"{best_key[0]} wire words covered, {best_key[1]:.0%} token match)")
    else:
        print("no confident issuer match -- using full-universe template "
              "(pass --issuer to override)")
    return best_name


def default_concession():
    """Delegates to the single calibration source (see calibration.py)."""
    from calibration import concession
    return concession()


def main():
    p = argparse.ArgumentParser(description="Price a new-issue wire against the model.")
    p.add_argument('--wire', required=True, help='path to the wire text file')
    p.add_argument('--evals', default=str(HERE / 'ICE_Evals.csv'))
    p.add_argument('--model', default=str(HERE / 'model.joblib'))
    p.add_argument('--state', default=None, help="e.g. 'California' (default: all states)")
    p.add_argument('--issuer', default=None, help='issuer text for the categorical template')
    p.add_argument('--template-cusip', default=None, help='use this CUSIP as the template instead')
    p.add_argument('--concession', type=float, default=None,
                   help='new-issue concession, bps (default: auto-calibrated from wire_archive)')
    p.add_argument('--settlement', default=None, help="settlement date m/d/yyyy if the wire has no DATED line")
    p.add_argument('--retrain', action='store_true')
    p.add_argument('--report', action='store_true')
    p.add_argument('--no-archive', action='store_true',
                   help='skip writing to wire_archive (for tests/dry runs)')
    args = p.parse_args()
    if args.concession is None:
        args.concession = default_concession()

    # ----- model: load cache or train -----
    model_path = Path(args.model)
    cache_path = model_path.with_name('template_cache.parquet')
    df = None
    if model_path.exists() and not args.retrain:
        print(f"loading cached model {model_path.name} (use --retrain after a new eval pull)")
        bundle = mm.load_bundle(model_path)
    else:
        print("training (canonical stacked recipe)...")
        df = mm.stacked_frame(HERE)
        bundle = mm.train_yield_model(df)
        if args.report:
            mm.error_report(bundle)
        mm.save_bundle(bundle, model_path)
        # slim per-bond cache so later pricing runs skip the big evals CSV
        tmpl_cols = [c for c in mm.CATEGORICAL_FEATURES
                     + ['primary_name_abbreviated'] if c in df.columns]
        df[tmpl_cols].to_parquet(cache_path)
        print(f"template cache -> {cache_path.name}")

    # ----- parse the wire first (issuer auto-match needs the description) -----
    text = open(args.wire, encoding='utf-8').read()
    deal = parse_wire(text)

    # ----- template categoricals (from slim cache when available) -----
    if df is None:
        if cache_path.exists():
            import pandas as pd
            df = pd.read_parquet(cache_path)
        else:
            df = mm.load_evals(args.evals)
            df = mm.clean_universe(df, state=args.state)
    issuer = args.issuer or auto_issuer(deal['description'], df)
    template = mm.template_from(df, issuer_contains=issuer, cusip=args.template_cusip)

    # credit-consistency guard: if the wire's rating is far from the matched
    # issuer's typical rating, this deal is a DIFFERENT credit of that issuer
    # (e.g. a BBB toll deal from an issuer whose outstanding bonds are AA
    # sales-tax). Keep the issuer's state/sector context but drop the
    # issuer-identity anchors and rating fields, which would otherwise pull
    # every tranche toward the wrong credit.
    import pandas as _pd
    tmpl_moody = _pd.to_numeric(_pd.Series([template.get('normalized_moody_long_rating')]),
                                errors='coerce').iloc[0]
    if (deal.get('moody_rating') is not None and _pd.notna(tmpl_moody)
            and abs(deal['moody_rating'] - tmpl_moody) >= 3 and not args.template_cusip):
        # a new credit has no true comparable in the issuer's own book; the
        # honest context is the LOCALITY (all issuers sharing the name's
        # geographic prefix) with the wire's rating layered on by price_wire.
        broad = ' '.join((issuer or '').split()[:2])
        print(f"credit mismatch (wire {deal['moody_rating']} vs pool {tmpl_moody:.0f}): "
              f"new credit of this issuer -- broadening template to '{broad}' "
              f"(pass --template-cusip of a true comparable for precision)")
        template = mm.template_from(df, issuer_contains=broad)

    print()
    print(deal['description'])
    size = f"${deal['issue_amount']:,}" if deal['issue_amount'] else "size not found on wire"
    print(f"deal {size} | Moody's {deal['moody']} / S&P {deal['sandp']} "
          f"/ Fitch {deal['fitch']} | dated {deal['dated_date']} | "
          f"call {deal['call_date']} @ {deal['call_price']} | {len(deal['tranches'])} tranches\n")

    results = mm.price_wire(deal, bundle, template,
                            settlement_date=args.settlement,
                            concession_bps=args.concession)
    # stored so concession_tracker.py can back out each deal's implied
    # concession from the archive later
    results['Concession Used (bps)'] = args.concession
    print()
    print(results.to_string())

    # ----- archive -----
    if args.no_archive:
        print("\n(--no-archive: result not written to wire_archive)")
        return
    arch_dir = HERE / 'wire_archive'
    arch_dir.mkdir(exist_ok=True)
    stamp = datetime.now().strftime('%m-%d-%Y_%H-%M-%S')
    deal_tag = ''.join(ch for ch in deal['description'][:40] if ch.isalnum() or ch in ' -').strip().replace(' ', '_')
    out = arch_dir / f"{deal_tag}_{stamp}.csv"
    results.to_csv(out)
    print(f"\narchived -> {out}")


if __name__ == '__main__':
    main()


In [ ]:
%%writefile train_production.py
"""Train the production ensemble (current muni_model defaults) and save
model.joblib + template_cache.parquet. Writes train_production.done with the
holdout metrics when finished -- used by unattended/detached runs."""
import sys
import traceback
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
import muni_model as mm

DONE = HERE / 'train_production.done'
DONE.unlink(missing_ok=True)

try:
    df = mm.stacked_frame(HERE)
    bundle = mm.train_yield_model(df)
    mm.save_bundle(bundle, HERE / 'model.joblib')
    cols = [c for c in mm.CATEGORICAL_FEATURES + ['primary_name_abbreviated'] if c in df.columns]
    df[cols].to_parquet(HERE / 'template_cache.parquet')
    DONE.write_text(f"OK mae={bundle['mae_bps']:.2f} median={bundle['median_bps']:.2f}\n")
    print("done")
except Exception:
    DONE.write_text("FAILED\n" + traceback.format_exc())
    raise


In [ ]:
%%writefile concession_tracker.py
"""
concession_tracker -- learn the new-issue concession from the deal archive.

    python concession_tracker.py

Each archived deal's implied concession = concession used at run time minus
the deal's mean signed error (if the model came out 3 bps below the wire on
average with 13 applied, the deal's true concession was ~16). Reports the
per-deal history and the recommended estimate + spread for rank_deals.py's
SIGMA_CONC. Gets sharper with every wire priced -- this is the cheapest
accuracy gain in the system.
"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path(__file__).resolve().parent
ARCH = HERE / 'wire_archive'

# implied concessions from the single calibration source (workout-axis
# definition, one vote per deal -- see calibration.py)
from calibration import _implied_per_deal, _latest_per_deal

implied = _implied_per_deal()
files = _latest_per_deal()
rows = []
for tag, val in sorted(implied.items()):
    d = pd.read_csv(files[tag])
    used = (float(d['Concession Used (bps)'].iloc[0])
            if 'Concession Used (bps)' in d.columns else float('nan'))
    rows.append({'deal': tag[:44], 'tranches': len(d),
                 'concession used': used,
                 'implied concession (bps)': round(val, 1)})

if not rows:
    sys.exit("no archived deals found in wire_archive\\")

t = pd.DataFrame(rows)
print(t.to_string(index=False))

imp = t['implied concession (bps)']
est, spread = imp.mean(), imp.std(ddof=1) if len(imp) > 1 else 3.0
print(f"\nacross {len(t)} deal(s): recommended concession {est:.0f} bps"
      f" | spread {spread:.1f} bps")
print(f"use:  run_pipeline.py --concession {est:.0f}")
if len(t) >= 5:
    print("5+ deals: consider splitting the estimate by rating tier "
          "(IG vs BBB priced differently in the first two already).")
else:
    print(f"({len(t)} deals is a thin estimate -- it tightens with every wire priced)")


In [ ]:
%%writefile test_suite.py
"""
test_suite -- verification of the wire-pricing pipeline.

    python test_suite.py

Checks, in order:
  1. 30/360 day-count math against hand-computed cases
  2. BAML wire parse: every deal-level field and spot-checked tranches
     against values read directly off the wire text
  3. Loop Capital (NYC TFA) wire parse: same treatment, other dialect
  4. Edge cases: no call features, maturity == call date, missing price line
  5. Rating canonicalization: wire-supplied ratings must reach the model
     (not silently become missing)
  6. Model determinism: same input twice -> identical predictions
  7. End-to-end regression: both wires price with sane error vs dealer

Exits non-zero on any failure. Run after ANY code change.
"""

import sys
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))

from Wire_Parser import parse_wire  # noqa: E402
import muni_model as mm  # noqa: E402

FAILURES = []


def check(name, cond, detail=''):
    status = 'ok  ' if cond else 'FAIL'
    print(f"  [{status}] {name}" + (f"  ({detail})" if detail and not cond else ''))
    if not cond:
        FAILURES.append(f"{name}: {detail}")


# ------------------------------------------------------------ 1. day count
print("1. 30/360 day count")
check("9/3/26 -> 6/1/36 = 3508", mm.days_30_360('09/03/2026', '06/01/2036') == 3508)
check("1/31 -> 7/31 = 180 (both EOM)", mm.days_30_360('01/31/2026', '07/31/2026') == 180)
check("1/30 -> 2/28 = 28", mm.days_30_360('01/30/2026', '02/28/2026') == 28)
check("same day = 0", mm.days_30_360('06/01/2036', '06/01/2036') == 0)
check("9/3/26 -> 6/1/45 = 6748", mm.days_30_360('09/03/2026', '06/01/2045') == 6748)

# ------------------------------------------------------------ 2. BAML wire
print("2. BAML wire parse")
baml = parse_wire(open(HERE / 'BAML Write-Up.txt', encoding='utf-8').read())
check("deal size 509,145,000", baml['issue_amount'] == 509_145_000)
check("16 tranches", len(baml['tranches']) == 16)
check("tranche sum == deal size", sum(t['amount'] for t in baml['tranches']) == 509_145_000)
check("Moody's Baa2 -> 9", baml['moody'] == 'Baa2' and baml['moody_rating'] == 9)
check("S&P NR falls back to Fitch BBB- -> 10", baml['sandp'] == 'NR' and baml['sandp_rating'] == 10)
check("dated 09/03/2026", baml['dated_date'] == '09/03/2026')
check("call 06/01/2036 @ 100", baml['call_date'] == '06/01/2036' and baml['call_price'] == 100.0)

t34 = baml['tranches'][0]
check("2034: 5,650M / 5% / 3.57 / 109.595 / no PTC",
      t34['maturity'] == '06/01/2034' and t34['amount'] == 5_650_000
      and t34['coupon'] == 5.0 and t34['yield'] == 3.57
      and t34['price'] == 109.595 and t34['ptc_date'] is None)
t44 = next(t for t in baml['tranches'] if t['maturity'] == '06/01/2044')
check("2044: 17,965M / 4.54 / 103.583 / PTC 06/01/2036",
      t44['amount'] == 17_965_000 and t44['yield'] == 4.54
      and t44['price'] == 103.583 and t44['ptc_date'] == '06/01/2036')
t56s = [t for t in baml['tranches'] if t['maturity'] == '06/01/2056']
check("two 2056 tranches (5.0 disc / 5.25 PTC)",
      len(t56s) == 2
      and {t['coupon'] for t in t56s} == {5.0, 5.25}
      and next(t for t in t56s if t['coupon'] == 5.0)['price'] == 98.319
      and next(t for t in t56s if t['coupon'] == 5.0)['ptc_date'] is None
      and next(t for t in t56s if t['coupon'] == 5.25)['price'] == 101.439)
check("takedown parsed 0.40", all(t['takedown'] == 0.40 for t in baml['tranches']))
check("description has issuer", 'LOS ANGELES' in baml['description'])

# ------------------------------------------------------------ 3. NYC TFA wire
print("3. Loop Capital (NYC TFA) wire parse")
nyc = parse_wire(open(HERE / 'NYC_TFA_wire.txt', encoding='utf-8').read())
check("deal size 1,500,000,000 from Subject", nyc['issue_amount'] == 1_500_000_000)
check("7 tranches despite orders/spread tokens", len(nyc['tranches']) == 7)
check("tranche par sum 926,405,000",
      sum(t['amount'] for t in nyc['tranches']) == 926_405_000)
t46 = nyc['tranches'][0]
check("2046: 84,630M / 5.25 / 4.57 / 105.478 / PTC 11/01/2036",
      t46['amount'] == 84_630_000 and t46['coupon'] == 5.25
      and t46['yield'] == 4.57 and t46['price'] == 105.478
      and t46['ptc_date'] == '11/01/2036')
t53 = next(t for t in nyc['tranches'] if t['coupon'] == 5.0)
check("2053 5.00%: 122,075M / 4.93 / 100.547",
      t53['maturity'] == '11/01/2053' and t53['amount'] == 122_075_000
      and t53['yield'] == 4.93 and t53['price'] == 100.547)
check("no ratings on wire -> None (template will supply)",
      nyc['moody_rating'] is None and nyc['sandp_rating'] is None)
check("call 11/01/2036 @ 100", nyc['call_date'] == '11/01/2036' and nyc['call_price'] == 100.0)
check("no DATED line -> None", nyc['dated_date'] is None)
check("description from Subject", 'Transitional' in nyc['description'])

# ------------------------------------------------------------ 4. edge cases
print("4. edge cases")
no_call = parse_wire("""RE: $ 10,000,000
TEST ISSUER
MOODY'S: Aa2 (Stable)
DATED:01/15/2027
06/01/2030     10,000M     5.00%     3.00      0.25
                      (Approx. $ Price 106.100)
""")
check("no CALL FEATURES -> call_date None", no_call['call_date'] is None)
check("single tranche parsed", len(no_call['tranches']) == 1
      and no_call['tranches'][0]['price'] == 106.100)
check("Aa2 -> 3", no_call['moody_rating'] == 3)

at_call = parse_wire("""RE: $ 5,000,000
TEST
DATED:01/15/2027
06/01/2036      5,000M     5.00%     3.50      0.25
                      (Approx. $ Price 104.000)
CALL FEATURES:  Optional call in 06/01/2036 @ 100.00
""")
from datetime import datetime as _dt
mat_eq_call = (_dt.strptime(at_call['tranches'][0]['maturity'], '%m/%d/%Y')
               > _dt.strptime(at_call['call_date'], '%m/%d/%Y'))
check("maturity == call date -> NOT callable", mat_eq_call is False)

no_price = parse_wire("""RE: $ 5,000,000
TEST
DATED:01/15/2027
06/01/2030      5,000M     5.00%     3.00      0.25
""")
check("missing price line -> price None (skipped later, not crash)",
      no_price['tranches'][0]['price'] is None)

# ------------------------------------------------------------ 5. rating canonicalization
print("5. categorical canonicalization")
c = mm._canon_cat(pd.Series([9, '9', 9.0, 'True', np.nan, 'AA+']))
check("9 / '9' / 9.0 all -> '9.0'", list(c[:3]) == ['9.0', '9.0', '9.0'])
check("'True' stays 'True'", c.iloc[3] == 'True')
check("'AA+' stays 'AA+'", c.iloc[5] == 'AA+')

# ------------------------------------------------------------ 6/7. model + end-to-end
print("6. model training, determinism, rating reach")
import os
df = mm.load_evals(HERE / 'ICE_Evals.csv')
df = mm.clean_universe(df)
if os.environ.get('TEST_REUSE_MODEL') == '1':
    print("  (TEST_REUSE_MODEL=1: validating the existing model.joblib, no retrain)")
    bundle = mm.load_bundle(HERE / 'model.joblib')
else:
    # same canonical recipe as train_production -- a single-snapshot train
    # here once silently overwrote the stacked production model (8/27)
    bundle = mm.train_yield_model(mm.stacked_frame(HERE), seed=42)
    mm.save_bundle(bundle, HERE / 'model.joblib')
check("holdout MAE < 10 bps", bundle['mae_bps'] < 10, f"{bundle['mae_bps']:.1f}")

dfF = mm.build_features(df)
Xa = mm._prep_matrix(dfF.head(500), bundle['numeric'], bundle['categorical'], bundle['categories'])
p1, p2 = mm.predict_bundle(bundle, Xa), mm.predict_bundle(bundle, Xa)
check("deterministic predictions", bool(np.array_equal(p1, p2)))

reloaded = mm.load_bundle(HERE / 'model.joblib')
p3 = mm.predict_bundle(reloaded, mm._prep_matrix(dfF.head(500), reloaded['numeric'],
                                                 reloaded['categorical'], reloaded['categories']))
check("save/reload identical predictions", bool(np.allclose(p1, p3)))

# rating override must land in a real category, not NaN
row = pd.DataFrame([{'normalized_moody_long_rating': 9}])
Xr = mm._prep_matrix(row, [], ['normalized_moody_long_rating'], bundle['categories'])
check("wire rating 9 reaches model as '9.0' category",
      not pd.isna(Xr['normalized_moody_long_rating'].iloc[0]),
      "override became missing -- canonicalization broken")

print("7. end-to-end regression on both wires")
# broad template (production usage): the I-105 toll deal is a separate credit
# from MTA's sales-tax bonds, so a narrow same-issuer template anchors to the
# wrong credit (verified: ~33 bps one-directional skew). price_wire warns on
# that mismatch; the broad pool is correct here.
templ_la = mm.template_from(df, issuer_contains='LOS ANG')
res_la = mm.price_wire(baml, bundle, templ_la, concession_bps=13)  # pinned-for-regression: tests stay stable vs drifting calibration
la_mae = res_la['Error (bps)'].abs().mean()
check("LA Metro: 16 rows priced", len(res_la) == 16)
check("LA Metro mean abs err < 8 bps (was ~3)", la_mae < 8, f"{la_mae:.1f}")

templ_bad = mm.template_from(df, issuer_contains='LOS ANG CY CA MET TRA AUT')
import io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    mm.price_wire(baml, bundle, templ_bad, concession_bps=13)  # pinned-for-regression: tests stay stable vs drifting calibration
check("credit-mismatch template triggers warning", 'WARNING: wire rating' in buf.getvalue())

# an empty/placeholder wire must fail LOUDLY with guidance, not crash with
# a cryptic AttributeError (seen live 8/27: placeholder WIRE_TEXT cell)
empty_deal = parse_wire("PASTE THE ENTIRE WIRE MESSAGE HERE")
try:
    mm.price_wire(empty_deal, bundle, templ_la, concession_bps=13)  # pinned-for-regression
    check("empty wire raises a clear error", False, "no exception raised")
except ValueError as e:
    check("empty wire raises a clear error", 'No priced tranches' in str(e))
except Exception as e:
    check("empty wire raises a clear error", False, f"wrong exception: {type(e).__name__}")

templ_ny = mm.template_from(df, issuer_contains='CITY TRANSITIONAL FIN')
res_ny = mm.price_wire(nyc, bundle, templ_ny, concession_bps=13)  # pinned-for-regression: tests stay stable vs drifting calibration
ny_mae = res_ny['Error (bps)'].abs().mean()
check("NYC TFA: 7 rows priced", len(res_ny) == 7)
check("NYC TFA mean abs err < 8 bps (was ~3)", ny_mae < 8, f"{ny_mae:.1f}")
check("all predictions finite", bool(np.isfinite(res_la['Model Yield']).all()
                                     and np.isfinite(res_ny['Model Yield']).all()))

# ------------------------------------------------------------ verdict
print()
if FAILURES:
    print(f"{len(FAILURES)} FAILURE(S):")
    for f in FAILURES:
        print("  -", f)
    sys.exit(1)
print("ALL CHECKS PASSED")


## 3. Train the model (once per data refresh, ~20 min)
Needs `ICE_Evals.csv` in this folder. Writes `model.joblib` and `template_cache.parquet`.

In [ ]:
# %run train_production.py

## 4. Price a wire
Paste the FULL Bloomberg wire message between the triple quotes and run. (Or skip this cell and run `%run run_pipeline.py --wire yourfile.txt` on a saved file.)

In [ ]:
WIRE_TEXT = """
PASTE THE ENTIRE WIRE MESSAGE HERE
"""

open('pasted_wire.txt', 'w', encoding='utf-8').write(WIRE_TEXT)
%run run_pipeline.py --wire pasted_wire.txt

## 5. Concession tracker (run occasionally)

In [ ]:
%run concession_tracker.py

## 6. Verify the installation / any code change

In [ ]:
import os
os.environ['TEST_REUSE_MODEL'] = '1'
%run test_suite.py